Cross-Matching Multiple Source Catalogs in jdaviz

Treat the first catalog as the base. Match each subsequent catalog against the running merged set. This generalizes to N catalogs and keeps every step a well-understood pairwise problem.

Two matching modes per pair:
- sky — SkyCoord.match_to_catalog_sky + a separation tolerance (the recommended default, since source ids are rarely shared across catalogs from different instruments).
- id — exact join on a shared source-id column (fast, unambiguous when ids are global).
- id_then_sky — use ids where present, fall back to sky for the rest.

Ambiguity detection (the part that needs user confirmation):
- a nearest neighbor whose separation is in a "grey zone" (tolerance < sep <= review_radius),
- many-to-one collisions: two base sources whose best match is the same other-catalog source.
These are returned as a separate review table rather than silently accepted/rejected.

In [1]:
import numpy as np
from astropy import units as u
from astropy.table import QTable

from jdaviz.core.crossmatch import (crossmatch_pair, crossmatch_catalogs,
                                     apply_review_decisions)

rng = np.random.default_rng(42)


**WARNING**: LOCAL JWST PRD VERSION PRDOPSSOC-068 DOESN'T MATCH THE CURRENT ONLINE VERSION PRDOPSSOC-073
Please consider updating pysiaf, e.g. pip install --upgrade pysiaf or conda update pysiaf


1.1 Synthetic catalogs to exercise the engine
We build three catalogs around the same field:

A (base): 8 sources, with a global source_id.
B: same field, shares ids for some sources, positions jittered by ~0.3″, plus an extra source.
C: position-only (no shared ids), with one deliberately ambiguous pair to trigger review.

In [2]:
base_ra = 150.0 + rng.uniform(-0.01, 0.01, size=8)
base_dec = 2.0 + rng.uniform(-0.01, 0.01, size=8)
cat_a = QTable({
    'source_id': [f'A{i:03d}' for i in range(8)],
    'ra': base_ra * u.deg,
    'dec': base_dec * u.deg,
})

# B: jitter ~0.3 arcsec (~8.3e-5 deg), share ids for first 5 sources
jit = 0.3 / 3600.0
cat_b = QTable({
    'source_id': [f'A{i:03d}' for i in range(5)] + ['B900', 'B901', 'B902'],
    'ra': np.concatenate([base_ra[:5] + rng.uniform(-jit, jit, 5),
                          150.0 + rng.uniform(-0.01, 0.01, 3)]) * u.deg,
    'dec': np.concatenate([base_dec[:5] + rng.uniform(-jit, jit, 5),
                           2.0 + rng.uniform(-0.01, 0.01, 3)]) * u.deg,
})

# C: position only. Source 6 is placed in the grey zone (~1.5 arcsec) to force review.
grey = 1.5 / 3600.0
c_ra = base_ra.copy()
c_dec = base_dec.copy()
c_ra[6] += grey
cat_c = QTable({
    'Right Ascension (degrees)': c_ra * u.deg,
    'Declination (degrees)': c_dec * u.deg,
})

catalogs = [('A', cat_a), ('B', cat_b), ('C', cat_c)]
len(cat_a), len(cat_b), len(cat_c)

(8, 8, 8)

In [3]:
catalogs

[('A',
  <QTable length=8>
  source_id         ra                dec        
                   deg                deg        
     str4        float64            float64      
  --------- ------------------ ------------------
       A000 150.00547912097113 1.9925622726535108
       A001 149.99877756879505 1.9990077187579114
       A002 150.00717195839823 1.9974159604846515
       A003 150.00394736058118  2.008535299776972
       A004 149.99188354695775 2.0028773024016133
       A005 150.00951244703273 2.0064552322654166
       A006  150.0052227940398 1.9988682839765466
       A007 150.00572128610554 1.9945447744356954),
 ('B',
  <QTable length=8>
  source_id         ra                dec        
                   deg                deg        
     str4        float64            float64      
  --------- ------------------ ------------------
       A000 150.00548821843563 1.9926086699030232
       A001 149.99870487167107   1.99895682520922
       A002 150.00722656359358 1.99741041398

In [ ]:
merged, review = crossmatch_catalogs(
    catalogs,
    tolerance=1 * u.arcsec,
    review_radius=2 * u.arcsec,
    id_columns={'A': 'source_id', 'B': 'source_id'},
    mode='id_then_sky',
)
merged

base_idx,object_id,base_ra,base_dec,A_idx,B_idx,B_sep_arcsec,B_status,C_idx,C_sep_arcsec,C_status,match_count,needs_review
,,deg,deg,,,,,,,,,
int64,object,float64,float64,int64,int64,float64,object,int64,float64,object,int64,bool
0,A000,150.00547912097113,1.9925622726535108,0,0,0.17020686410955332,matched,0,0.0,matched,3,False
1,A001,149.99877756879505,1.9990077187579114,1,1,0.3193383614879235,matched,1,0.0,matched,3,False
2,A002,150.00717195839823,1.9974159604846515,2,2,0.1974713618111863,matched,2,0.0,matched,3,False
3,A003,150.00394736058118,2.008535299776972,3,3,0.28487632528196527,matched,3,0.0,matched,3,False
4,A004,149.99188354695775,2.0028773024016133,4,4,0.2587966843260676,matched,4,0.0,matched,3,False
5,A005,150.00951244703273,2.0064552322654166,5,-1,nan,unmatched,5,0.0,matched,2,False
6,A006,150.0052227940398,1.9988682839765466,6,-1,nan,unmatched,6,1.4990872742460732,review,2,True
7,A007,150.00572128610554,1.9945447744356954,7,-1,nan,unmatched,7,0.0,matched,2,False


In [5]:
print('Rows needing user confirmation:', len(review))
review

Rows needing user confirmation: 1


base_idx,object_id,base_ra,base_dec,A_idx,B_idx,B_sep_arcsec,B_status,C_idx,C_sep_arcsec,C_status,match_count,needs_review
,,deg,deg,,,,,,,,,
int64,object,float64,float64,int64,int64,float64,object,int64,float64,object,int64,bool
6,A006,150.0052227940398,1.9988682839765466,6,-1,nan,unmatched,6,1.4990872742460732,review,2,True


1.2 Applying a user decision
The review table is what gets surfaced in the UI. The user accepts/rejects each flagged candidate; the helper below applies those decisions back onto the merged table.

In [6]:
# Example: user confirms C matches base source 6 after all.
decisions = {6: {'C': True}}
resolved = apply_review_decisions(merged, decisions)
resolved[['base_idx', 'A_idx', 'B_idx', 'C_idx', 'match_count', 'needs_review']]


base_idx,A_idx,B_idx,C_idx,match_count,needs_review
int64,int64,int64,int64,int64,bool
0,0,0,0,3,False
1,1,1,1,3,False
2,2,2,2,3,False
3,3,3,3,3,False
4,4,4,4,3,False
5,5,-1,5,2,False
6,6,-1,6,2,False
7,7,-1,7,2,False
8,-1,5,-1,1,False


A tray plugin under category data:analysis, sitting next to Catalog Search. It can match any set of already-loaded catalogs, re-run interactively, and own the review UI. Add a metadata-association mode that groups loaded data by a chosen metadata key (meta['OBJECT'], proposal id, etc.) and/or sky footprint overlap.

### Suggested ticket order
1. Create jdaviz/core/crossmatch.py by building on the previous code. Also include unit tests.
2. Add the Catalog Cross-Match plugin and connect it to the crossmatch code
3. Add the review/confirmation sub-panel
4. Add the ability to link spectrum/2D spectrum/cube to a row so that data is populated